# Notebook Role: Phase 3 – Feature Conversion & Preparation

This notebook converts raw text into multiple feature variants:
- **TF‑IDF (raw and chi2‑selected)**
- **Lexicon (VADER sentiment scores)**
- **Hybrid (TF‑IDF + lexicon)**

Outputs:
- `X_train_*` / `X_val_*` feature matrices
- `labels_train.csv` / `labels_val.csv`
- `tfidf_vectorizer.pkl`, `chi2_selector.pkl`
- `hybrid_features_metadata.json`

Training set: `Phase_1/master_reviews.csv`  
Validation set: `Phase_2/mock_pseudo_labeled.csv`

This notebook hardens preprocessing so all models in Phase 5 consume consistent artifacts.


In [1]:
# Imports
import pandas as pd
import numpy as np
import os
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import LabelEncoder
import joblib
from scipy import sparse
import json

# Ensure resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("vader_lexicon")

# Input paths
file_master = "Phase_1/master_reviews.csv"
file_mock   = "Phase_2/mock_pseudo_labeled.csv" 

# Output directory
output_dir = "Phase_3"
os.makedirs(output_dir, exist_ok=True)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
# Load datasets
df_train = pd.read_csv(file_master)
df_val   = pd.read_csv(file_mock)

print("Training rows:", len(df_train))
print("Validation rows:", len(df_val))


Training rows: 409577
Validation rows: 7486680


In [ ]:
import re
import swifter
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

# Extend with domain-specific stopwords
custom_stopwords = stop_words.union({
    "day", "month", "year", "week", "mg", "pill", "tablet", "dose",
    "time", "period", "started", "taking", "get", "take", "first",
    "november", "nov", "noticing"
})

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in custom_stopwords]
    return " ".join(tokens)

# Swifter automatically decides whether to parallelize/vectorize
df_train["text_cleaned"] = df_train["ReviewText"].swifter.apply(clean_text)
df_val["text_cleaned"]   = df_val["ReviewText"].swifter.apply(clean_text)


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    max_df=0.95,
    min_df=5
)

X_train_tfidf = vectorizer.fit_transform(df_train["text_cleaned"])
X_val_tfidf   = vectorizer.transform(df_val["text_cleaned"])

y_train = df_train["Sentiment"]
y_val   = df_val["Sentiment"]

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)


In [ ]:
vader = SentimentIntensityAnalyzer()

def lexicon_features(text):
    scores = vader.polarity_scores(text)
    return [scores["pos"], scores["neg"], scores["neu"], scores["compound"]]

lex_train = np.array([lexicon_features(t) for t in df_train["text_cleaned"]])
lex_val   = np.array([lexicon_features(t) for t in df_val["text_cleaned"]])

print("Lexicon feature shape (train):", lex_train.shape)
print("Lexicon feature shape (val):", lex_val.shape)


In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

selector = SelectKBest(chi2, k=3000)
X_train_selected = selector.fit_transform(X_train_tfidf, y_train_enc)
X_val_selected   = selector.transform(X_val_tfidf)

print("Selected TF-IDF training shape:", X_train_selected.shape)
print("Selected TF-IDF validation shape:", X_val_selected.shape)


In [ ]:
from scipy.sparse import hstack, vstack, csr_matrix

def safe_hybrid_stack(X_tfidf, lex_array, batch_size=500_000):
    """Safely stack sparse TF-IDF with dense lexicon features."""
    lex_sparse = csr_matrix(lex_array)
    if X_tfidf.shape[0] <= batch_size:
        return hstack([X_tfidf, lex_sparse])
    
    # Chunked stacking for large sets
    chunks = []
    for i in range(0, X_tfidf.shape[0], batch_size):
        tfidf_chunk = X_tfidf[i:i+batch_size]
        lex_chunk   = lex_sparse[i:i+batch_size]
        chunks.append(hstack([tfidf_chunk, lex_chunk]))
    return vstack(chunks)

# Apply to train and val
X_train_hybrid = safe_hybrid_stack(X_train_selected, lex_train)
X_val_hybrid   = safe_hybrid_stack(X_val_selected, lex_val)

print("Hybrid training shape:", X_train_hybrid.shape)
print("Hybrid validation shape:", X_val_hybrid.shape)


In [ ]:
# Save hybrid artifacts
sparse.save_npz(os.path.join(output_dir, "X_train_hybrid.npz"), X_train_hybrid)
df_train[["ReviewID","Sentiment"]].to_csv(os.path.join(output_dir, "labels_train.csv"), index=False)

sparse.save_npz(os.path.join(output_dir, "X_val_hybrid.npz"), X_val_hybrid)
df_val[["ReviewID","Sentiment"]].to_csv(os.path.join(output_dir, "labels_val.csv"), index=False)

# Save vectorizer and selector
joblib.dump(vectorizer, os.path.join(output_dir, "tfidf_vectorizer.pkl"))
joblib.dump(selector, os.path.join(output_dir, "chi2_selector.pkl"))


# Save metadata
metadata = {
    "training_rows": len(df_train),
    "validation_rows": len(df_val),
    "features": X_train_hybrid.shape[1],
    "lexicon_features": ["pos", "neg", "neu", "compound"],
    "training_sentiment_distribution": y_train.value_counts().to_dict(),
    "validation_sentiment_distribution": y_val.value_counts().to_dict()
}
with open(os.path.join(output_dir, "hybrid_features_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Hybrid artifacts saved in Phase_3/")


In [ ]:
# Save raw TF-IDF artifacts
sparse.save_npz(os.path.join(output_dir, "X_train_tfidf.npz"), X_train_tfidf)
sparse.save_npz(os.path.join(output_dir, "X_val_tfidf.npz"), X_val_tfidf)
joblib.dump(vectorizer, os.path.join(output_dir, "tfidf_vectorizer.pkl"))


In [ ]:
# Save selected TF-IDF artifacts
sparse.save_npz(os.path.join(output_dir, "X_train_selected.npz"), X_train_selected)
sparse.save_npz(os.path.join(output_dir, "X_val_selected.npz"), X_val_selected)
joblib.dump(selector, os.path.join(output_dir, "chi2_selector.pkl"))


In [ ]:
# Save lexicon-only artifacts
sparse.save_npz(os.path.join(output_dir, "X_train_lexicon.npz"), sparse.csr_matrix(lex_train))
sparse.save_npz(os.path.join(output_dir, "X_val_lexicon.npz"), sparse.csr_matrix(lex_val))


In [ ]:
metadata.update({
    "feature_variants": {
        "raw_tfidf": X_train_tfidf.shape[1],
        "selected_tfidf": X_train_selected.shape[1],
        "lexicon_only": lex_train.shape[1],
        "hybrid": X_train_hybrid.shape[1]
    }
})
with open(os.path.join(output_dir, "hybrid_features_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)
